In [1]:
from __future__ import annotations
import logging
import hashlib
import os

from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Literal
from uuid import uuid4

from dotenv import load_dotenv
from pydantic import BaseModel, Field



In [2]:
!rm -rf chroma_db/
!rm -f data/processed/*.json
!rm -f eval/eval_set.json
!rm -rf eval/results/

In [5]:
## resolve project root directory
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(PROJECT_ROOT / ".env")


False

In [6]:
PROJECT_ROOT

PosixPath('/home/thimu/github_vs/protoRAG/rag-pipeline')

In [12]:
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)-7s | %(name)s | %(message)s",
)
# Add after logging.basicConfig in Cell 1
logging.getLogger("transformers").setLevel(logging.CRITICAL)
log = logging.getLogger("rag")

In [13]:
def _path(env_key: str, default: str) -> Path:
    """Resolve env-provided paths relative to project root unless absolute."""
    p = Path(os.getenv(env_key, default))
    return p.resolve() if p.is_absolute() else (PROJECT_ROOT / p).resolve()

In [14]:
class Config:
    # ollama
    OLLAMA_HOST: str = os.getenv("OLLAMA_HOST", "localhost")
    OLLAMA_PORT: int = int(os.getenv("OLLAMA_PORT", 11434))
    OLLAMA_MODEL: str = os.getenv("OLLAMA_MODEL", "gemma-4-e4b:latest")
    EMBEDDING_MODEL: str = os.getenv("EMBEDDING_MODEL", "embeddinggemma:latest")

    @property
    def OLLAMA_BASE_URL(self) -> str:
        return f"http://{self.OLLAMA_HOST}:{self.OLLAMA_PORT}"
    
    # Generation
    LLM_TEMPERATURE: float = float(os.getenv("LLM_TEMPERATURE", 0.0))
    LLM_NUM_CTX: int = int(os.getenv("LLM_NUM_CTX", 8192))

    # Retrieval
    CHUNK_SIZE: int = int(os.getenv("CHUNK_SIZE", 800))
    CHUNK_OVERLAP: int = int(os.getenv("CHUNK_OVERLAP", 120))
    TOP_K: int = int(os.getenv("TOP_K", 5))

    # Paths:
    CHROMA_PERSIST_DIR: Path = _path("CHROMA_PERSIST_DIR", "chroma_db")
    DATA_RAW_DIR: Path = _path("DATA_RAW_DIR", "/home/thimu/Downloads/pdf_splitter/IPC")

    DATA_PROCESSED_DIR: Path = _path("DATA_PROCESSED_DIR", "data/processed")

cfg = Config()


In [15]:
cfg.CHROMA_PERSIST_DIR.mkdir(parents=True, exist_ok=True)
cfg.DATA_PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

In [16]:
log.info(f"ollama base url: {cfg.OLLAMA_BASE_URL}")
log.info(f"ollama model: {cfg.OLLAMA_MODEL}")
log.info(f"embedding model: {cfg.EMBEDDING_MODEL}")
log.info(f"chunk/overlap/k: {cfg.CHUNK_SIZE}/{cfg.CHUNK_OVERLAP}/{cfg.TOP_K}")
log.info(f"Project root: {PROJECT_ROOT}")


2026-05-27 11:40:01,796 - INFO    | rag | ollama base url: http://localhost:11434
2026-05-27 11:40:01,796 - INFO    | rag | ollama model: gemma-4-e4b:latest
2026-05-27 11:40:01,796 - INFO    | rag | embedding model: embeddinggemma:latest
2026-05-27 11:40:01,796 - INFO    | rag | chunk/overlap/k: 800/120/5
2026-05-27 11:40:01,797 - INFO    | rag | Project root: /home/thimu/github_vs/protoRAG/rag-pipeline


In [17]:
SourceFormat = Literal["pdf", "txt", "md", "html", "docx", "xlsx", "pptx", "csv", "json", "xml", "jsonl", "yaml", "yml", "parquet", "avro", "orc", "tsv", "log"]
ElementType = Literal["text", "table", "list", "code", "heading", "metadata", "other"]

class RagChunk(BaseModel):
    """The atomic unit flowing through retrieval. All parsers produce these."""

    # identity
    chunk_id: str = Field(default_factory=lambda: str(uuid4()))
    content_hash: str = ""

    # content
    text: str

    # provenance - these are citation
    source_path: str
    source_format: SourceFormat

    # optional structured metadata (parser-specific, all optional)
    page_number: int | None = None # pdf
    slide_number: int | None = None # pptx
    sheet_name: str | None = None # excel
    section_heading: str | None = None # general document structure
    section_title: str | None = None # pdf bookmarks, html headings, etc.
    row_range: tuple[int, int] | None = None # csv, excel tables

    element_type: ElementType = "text"

    created_at: datetime = Field(default_factory=lambda: datetime.now(timezone.utc))
    extra: dict[str, Any] = Field(default_factory=dict)

    def model_post_init(self, __context: Any) -> None:
        if not self.content_hash:
            self.content_hash = hashlib.sha256(self.text.encode("utf-8")).hexdigest()[:16]

    def to_langchain_metadata(self) -> dict[str, Any]:
        """Flat, JSON-safe, no-None dict for Chroma / LangChain Document.metadata"""
        md: dict[str, Any] = {
            "chunk_id": self.chunk_id,
            "content_hash": self.content_hash,
            "source_path": self.source_path,
            "source_format": self.source_format,
            "element_type": self.element_type,
            "created_at": self.created_at.isoformat(),
        }
        if self.page_number is not None:
            md["page_number"] = self.page_number
        if self.slide_number is not None:
            md["slide_number"] = self.slide_number
        if self.sheet_name is not None:
            md["sheet_name"] = self.sheet_name
        if self.section_heading is not None:
            md["section_heading"] = self.section_heading
        if self.section_title is not None:
            md["section_title"] = self.section_title
        if self.row_range is not None:
            md["row_range"] = f"{self.row_range[0]}-{self.row_range[1]}"
        return md

In [18]:
# smoke test
if __name__ == "__main__":
    _t = RagChunk(
        text="Section 3.2: All employees must complete annual compliance training.",
        source_path="data/raw/test.pdf",
        source_format="pdf",
        page_number=1,
        element_type="text",
    )
    log.info(f"Schema OK | id: {_t.chunk_id} | hash: {_t.content_hash}")
    log.info(f"Metadata Sample: {_t.to_langchain_metadata()}")

2026-05-27 11:40:02,845 - INFO    | rag | Schema OK | id: 0559ae45-bb46-4846-8eef-da68fa650fc2 | hash: 962912e80c914a92
2026-05-27 11:40:02,846 - INFO    | rag | Metadata Sample: {'chunk_id': '0559ae45-bb46-4846-8eef-da68fa650fc2', 'content_hash': '962912e80c914a92', 'source_path': 'data/raw/test.pdf', 'source_format': 'pdf', 'element_type': 'text', 'created_at': '2026-05-27T03:40:02.845555+00:00', 'page_number': 1}


In [19]:
## docling parser
from docling.document_converter import DocumentConverter
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [20]:
class DoclingParser:
    """Phase 0 parser for PDF/PPTX/DOCX/HTML/MD via IBM Docling.

    Pipeline: file → Docling DocumentConverter → markdown → recursive split → RagChunk[].
    Provenance at this stage is file-level (source_path). Page/slide-level
    provenance is added in Phase 1 with HybridChunker.
    """

    SUPPORTED: dict[str, SourceFormat] = {
    ".pdf": "pdf",
    ".pptx": "pptx",
    ".docx": "docx",
    ".html": "html",
    ".htm": "html",
    ".md": "md",
    }

    def __init__(self, chunk_size: int = 800, chunk_overlap: int = 120):
        self.converter = DocumentConverter()
        # Separators ordered from "strong semantic boundary" → "last resort".
        # Markdown headings come first so chunks rarely cross sections.
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n## ", "\n### ", "\n#### ", "\n\n", "\n", ". ", " ", ""],
        )

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"DoclingParser does not support {path.suffix}")

        source_format = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[Docling] parsing {path.name} ({source_format}) …")

        try:
            result = self.converter.convert(str(path))
        except Exception as e:
            log.error(f"[Docling] convert failed for {path.name}: {e}")
            return []

        markdown = result.document.export_to_markdown()
        if not markdown.strip():
            log.warning(f"[Docling] {path.name}: empty output")
            return []

        rel_source = self._relative_source(path)
        texts = self.splitter.split_text(markdown)

        chunks = [
            RagChunk(
                text=t,
                source_path=rel_source,
                source_format=source_format,
                element_type="text",
            )
            for t in texts if t.strip()
        ]
        log.info(f"[Docling] {path.name}: {len(chunks)} chunks")
        return chunks

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [21]:
Path

pathlib.Path

In [22]:
parser = DoclingParser(chunk_size=cfg.CHUNK_SIZE, chunk_overlap=cfg.CHUNK_OVERLAP)

# Find the first PDF/PPTX/DOCX in data/raw/
candidates: list[Path] = []
for ext in (".pdf", ".pptx", ".docx", ".md", ".html", ".htm", ):
    candidates.extend(cfg.DATA_RAW_DIR.rglob(f"*{ext}"))

if not candidates:
    log.warning("No PDF/PPTX/DOCX found under data/raw/ — drop one in and re-run")
else:
    sample = candidates[0]
    sample_chunks = parser.parse(sample)
    print(f"\nFile           : {sample.name}")
    print(f"Total chunks   : {len(sample_chunks)}")
    if sample_chunks:
        c = sample_chunks[0]
        print(f"\n--- First chunk ---")
        print(f"format         : {c.source_format}")
        print(f"length (chars) : {len(c.text)}")
        print(f"chunk_id       : {c.chunk_id[:8]}…")
        print(f"hash           : {c.content_hash}")
        print(f"\ntext preview:\n{c.text[:400]}")
        print(f"\nmetadata sample:\n{c.to_langchain_metadata()}")

2026-05-27 11:40:10,778 - INFO    | rag | [Docling] parsing IPC_OLD_split_2.pdf (pdf) …
2026-05-27 11:40:10,786 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-27 11:40:11,443 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
2026-05-27 11:40:12,619 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
Could not load the custom kernel for multi-scale deformable attention: Error building extension 'MultiScaleDeformableAttention': [1/2] /usr/bin/nvcc -MD -MF ms_deform_attn_cuda.cuda.o.d -DTORCH_EXTENSION_NAME=MultiScaleDeformableAttention -DTORCH_API_INCLUDE_EXTENSION_H -I/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/transformers/kernels/deformable_detr -isystem /home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/torch/include -isystem /home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/torch/include/torch/csrc/api/includ


File           : IPC_OLD_split_2.pdf
Total chunks   : 15

--- First chunk ---
format         : pdf
length (chars) : 620
chunk_id       : 769830db…
hash           : c9ce8ac455e8ef64

text preview:
## Illustration

A writes  his name  on the  back of  a bill  of exchange.  As the of this endorsement is to transfer the right to the bill to any who  may become  the lawful  holder of it, the endorsement is a security". effect person "valuable

31.

will". "A

31.  "A  will".--The  words  "a  will"  denote  any  testamentary document.

32.

referring to acts include illegal omissions. Words

32.

metadata sample:
{'chunk_id': '769830db-f6ed-4433-b79d-c57b7ae4bf49', 'content_hash': 'c9ce8ac455e8ef64', 'source_path': '/home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_2.pdf', 'source_format': 'pdf', 'element_type': 'text', 'created_at': '2026-05-27T03:40:24.328173+00:00'}


In [23]:
## structured data parser
import json
import pandas as pd


class StructuredDataParser:
    """Phase 0 parser for CSV/XLSX/JSON/JSONL/TXT.

    Tabular: row → chunk (configurable batch), context prepended.
    JSON: list-of-records → record-per-chunk; else pretty-printed + split.
    JSONL: line → record → chunk.
    TXT: recursive split.
    """

    SUPPORTED: dict[str, SourceFormat] = {
        ".csv": "csv",
        ".xlsx": "xlsx",
        ".xls": "xlsx",
        ".json": "json",
        ".jsonl": "jsonl",
        ".ndjson": "jsonl",
        ".txt": "txt",
    }

    def __init__(
        self,
        rows_per_chunk: int = 1,
        max_rows_warn: int = 10_000,
        chunk_size: int = 800,
        chunk_overlap: int = 120,
    ):
        self.rows_per_chunk = rows_per_chunk
        self.max_rows_warn = max_rows_warn
        self.splitter = RecursiveCharacterTextSplitter(
            chunk_size=chunk_size,
            chunk_overlap=chunk_overlap,
            separators=["\n\n", "\n", ". ", " ", ""],
        )

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"StructuredDataParser does not support {path.suffix}")

        fmt = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[Structured] parsing {path.name} ({fmt}) …")
        try:
            dispatch = {
                "csv": self._parse_csv,
                "xlsx": self._parse_xlsx,
                "json": self._parse_json,
                "jsonl": self._parse_jsonl,
                "txt": self._parse_txt,
            }
            return dispatch[fmt](path)
        except Exception as e:
            log.error(f"[Structured] failed on {path.name}: {e}")
            return []

    # ---------- CSV / XLSX ----------
    def _parse_csv(self, path: Path) -> list[RagChunk]:
        df = pd.read_csv(path)
        if len(df) > self.max_rows_warn:
            log.warning(f"{path.name}: {len(df)} rows — consider increasing rows_per_chunk")
        return self._rows_to_chunks(df, path, "csv", sheet_name=None)

    def _parse_xlsx(self, path: Path) -> list[RagChunk]:
        chunks: list[RagChunk] = []
        xl = pd.ExcelFile(path)
        for sheet in xl.sheet_names:
            df = xl.parse(sheet)
            if df.empty:
                continue
            chunks.extend(self._rows_to_chunks(df, path, "xlsx", sheet_name=sheet))
        return chunks

    def _rows_to_chunks(
        self,
        df: pd.DataFrame,
        path: Path,
        source_format: SourceFormat,
        sheet_name: str | None,
    ) -> list[RagChunk]:
        rel = self._relative_source(path)
        n = len(df)
        if n == 0:
            return []

        scope = f"File: {path.name}"
        if sheet_name:
            scope += f" | Sheet: {sheet_name}"

        chunks: list[RagChunk] = []
        for start in range(0, n, self.rows_per_chunk):
            end = min(start + self.rows_per_chunk, n)
            sub = df.iloc[start:end]
            row_blocks = [
                "\n".join(f"{col}: {self._format_value(val)}" for col, val in row.items())
                for _, row in sub.iterrows()
            ]
            text = f"{scope} | Rows: {start+1}-{end}\n\n" + "\n\n---\n\n".join(row_blocks)
            chunks.append(RagChunk(
                text=text,
                source_path=rel,
                source_format=source_format,
                sheet_name=sheet_name,
                row_range=(start + 1, end),
                element_type="table",
            ))
        return chunks

    # ---------- JSON / JSONL ----------
    def _parse_json(self, path: Path) -> list[RagChunk]:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)
        rel = self._relative_source(path)

        # List of dicts → record-per-chunk
        if isinstance(data, list) and data and all(isinstance(x, dict) for x in data):
            return [
                self._record_to_chunk(rec, idx, path, rel, "json")
                for idx, rec in enumerate(data)
            ]

        # Dict with single dominant list field → use that list
        if isinstance(data, dict):
            list_keys = [k for k, v in data.items() if isinstance(v, list) and len(v) > 1]
            if len(list_keys) == 1 and all(isinstance(x, dict) for x in data[list_keys[0]]):
                key = list_keys[0]
                return [
                    self._record_to_chunk(rec, idx, path, rel, "json", parent_key=key)
                    for idx, rec in enumerate(data[key])
                ]

        # Fallback: pretty-print whole document, split if large
        text = f"File: {path.name}\n\n{json.dumps(data, indent=2, ensure_ascii=False)}"
        parts = self.splitter.split_text(text) if len(text) > 2000 else [text]
        return [
            RagChunk(text=p, source_path=rel, source_format="json", element_type="text")
            for p in parts if p.strip()
        ]

    def _parse_jsonl(self, path: Path) -> list[RagChunk]:
        rel = self._relative_source(path)
        chunks: list[RagChunk] = []
        with path.open("r", encoding="utf-8") as f:
            for idx, line in enumerate(f):
                line = line.strip()
                if not line:
                    continue
                try:
                    rec = json.loads(line)
                except json.JSONDecodeError:
                    log.warning(f"{path.name}: line {idx+1} not valid JSON, skipping")
                    continue
                chunks.append(self._record_to_chunk(rec, idx, path, rel, "jsonl"))
        if len(chunks) > self.max_rows_warn:
            log.warning(f"{path.name}: {len(chunks)} chunks produced")
        return chunks

    def _record_to_chunk(
        self,
        rec: Any,
        idx: int,
        path: Path,
        rel: str,
        fmt: SourceFormat,
        parent_key: str | None = None,
    ) -> RagChunk:
        if isinstance(rec, dict):
            body = "\n".join(f"{k}: {self._format_value(v)}" for k, v in rec.items())
        else:
            body = json.dumps(rec, ensure_ascii=False)
        scope = f"File: {path.name}"
        if parent_key:
            scope += f" | Field: {parent_key}"
        scope += f" | Record: {idx+1}"
        return RagChunk(
            text=f"{scope}\n\n{body}",
            source_path=rel,
            source_format=fmt,
            row_range=(idx + 1, idx + 1),
            element_type="text",
        )

    # ---------- TXT ----------
    def _parse_txt(self, path: Path) -> list[RagChunk]:
        text = path.read_text(encoding="utf-8")
        if not text.strip():
            return []
        rel = self._relative_source(path)
        return [
            RagChunk(text=t, source_path=rel, source_format="txt", element_type="text")
            for t in self.splitter.split_text(text) if t.strip()
        ]

    # ---------- Helpers ----------
    @staticmethod
    def _format_value(val: Any) -> str:
        if val is None or (isinstance(val, float) and pd.isna(val)):
            return ""
        if isinstance(val, (dict, list)):
            return json.dumps(val, ensure_ascii=False)
        return str(val)

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [29]:
struct_parser = StructuredDataParser(
    rows_per_chunk=1,
    chunk_size=cfg.CHUNK_SIZE,
    chunk_overlap=cfg.CHUNK_OVERLAP,
)

candidates: list[Path] = []
for ext in (".csv", ".xlsx", ".json", ".jsonl", ".txt", ".ndjson", ".yaml", ".yml", ".parquet", ".avro", ".orc", ".tsv", ".log", ):
    candidates.extend(cfg.DATA_RAW_DIR.rglob(f"*{ext}"))

if not candidates:
    log.warning("No CSV/XLSX/JSON/JSONL/TXT/NDJSON/YAML/YML/PARQUET/AVRO/ORC/TSV/LOG found under data/raw/ — drop a few in to test")
else:
    for sample in candidates[:]:  # preview up to 5 files
        result = struct_parser.parse(sample)
        print(f"\n=== {sample.relative_to(PROJECT_ROOT)} ===")
        print(f"  chunks   : {len(result)}")
        if result:
            c = result[0]
            print(f"  format   : {c.source_format} | element: {c.element_type}")
            print(f"  preview  : {c.text[:300]!r}")
            print(f"  metadata : {c.to_langchain_metadata()}")

2026-05-27 11:44:31,583 - WARNING | rag | No CSV/XLSX/JSON/JSONL/TXT/NDJSON/YAML/YML/PARQUET/AVRO/ORC/TSV/LOG found under data/raw/ — drop a few in to test


In [30]:
# [█████░░░░░░░░░░░░░░░░░░░░░░░░░░░░] ~15%

# PHASE 0 — Naive RAG ◄── WE ARE HERE
#    ✓ Config + schema
#    ✓ Parsers (Docling track + structured track)
#    ◯ Parser dispatcher          ← next (Piece 5)
#    ◯ Embedding + Chroma vector store
#    ◯ Top-k retrieval
#    ◯ Generation with Ollama (gemma-4-e4b)
#    ◯ End-to-end answer with citations

# PHASE 1 — Hybrid Retrieval
# PHASE 2 — Query Intelligence
# PHASE 3 — Agentic Orchestration (LangGraph)
# PHASE 4 — Evaluation & Observability
# PHASE 5 — Production Hardening (FastAPI service)
# PHASE 6 — Cloud & Scale (Docker, Azure/AWS/GCP)

In [31]:
from collections import defaultdict
from itertools import groupby
from tqdm.auto import tqdm


class ParserDispatcher:
    """Routes files to the right parser; aggregates RagChunks across a corpus.

    Extension point: append new parsers to `parsers` — first match wins.
    """

    def __init__(self, parsers: list):
        if not parsers:
            raise ValueError("At least one parser required")
        self.parsers = parsers

    def parse_file(self, path: Path) -> list[RagChunk]:
        for parser in self.parsers:
            if parser.supports(path):
                return parser.parse(path)
        return []

    def parse_directory(
        self,
        directory: Path,
        recursive: bool = True,
    ) -> list[RagChunk]:
        if not directory.exists():
            raise FileNotFoundError(f"Directory not found: {directory}")

        files = self._discover_files(directory, recursive)
        if not files:
            log.warning(f"No supported files under {directory}")
            return []

        log.info(f"Found {len(files)} supported file(s) under {directory.name}/")
        all_chunks: list[RagChunk] = []
        files_per_fmt: dict[str, int] = defaultdict(int)
        chunks_per_fmt: dict[str, int] = defaultdict(int)
        failed: list[str] = []
        seen_hashes: set[str] = set()
        dup_count = 0

        for fp in tqdm(files, desc="Parsing", unit="file"):
            try:
                chunks = self.parse_file(fp)
            except Exception as e:
                log.error(f"Parse error on {fp.name}: {e}")
                failed.append(fp.name)
                continue
            if not chunks:
                continue

            fmt = chunks[0].source_format
            files_per_fmt[fmt] += 1
            for chunk in chunks:
                if chunk.content_hash in seen_hashes:
                    dup_count += 1
                    continue
                seen_hashes.add(chunk.content_hash)
                all_chunks.append(chunk)
                chunks_per_fmt[fmt] += 1

        self._print_summary(files_per_fmt, chunks_per_fmt, failed, dup_count, len(all_chunks))
        return all_chunks

    def _discover_files(self, directory: Path, recursive: bool) -> list[Path]:
        pattern = "**/*" if recursive else "*"
        files: list[Path] = []
        for p in directory.glob(pattern):
            if not p.is_file():
                continue
            if any(part.startswith(".") for part in p.parts):
                continue
            if any(parser.supports(p) for parser in self.parsers):
                files.append(p)
        return sorted(files)

    @staticmethod
    def _print_summary(files_per_fmt, chunks_per_fmt, failed, dup_count, total) -> None:
        log.info("=" * 56)
        log.info(f"{'Format':<10} {'Files':>10} {'Chunks':>12}")
        log.info("-" * 56)
        for fmt in sorted(files_per_fmt):
            log.info(f"{fmt:<10} {files_per_fmt[fmt]:>10} {chunks_per_fmt[fmt]:>12}")
        log.info("-" * 56)
        log.info(f"Total unique chunks : {total}")
        log.info(f"Duplicates skipped  : {dup_count}")
        if failed:
            shown = failed[:5]
            extra = f" (+{len(failed)-5} more)" if len(failed) > 5 else ""
            log.info(f"Failed files        : {len(failed)} — {shown}{extra}")
        log.info("=" * 56)


# --- Cache helpers (so we don't re-parse on every notebook re-run) ---

def save_chunks_cache(chunks: list[RagChunk], path: Path) -> None:
    payload = [c.model_dump(mode="json") for c in chunks]
    path.write_text(
        json.dumps(payload, indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    log.info(f"Cached {len(chunks)} chunks → {path.relative_to(PROJECT_ROOT)}")


def load_chunks_cache(path: Path) -> list[RagChunk]:
    if not path.exists():
        return []
    data = json.loads(path.read_text(encoding="utf-8"))
    chunks = [RagChunk(**item) for item in data]
    log.info(f"Loaded {len(chunks)} chunks ← {path.relative_to(PROJECT_ROOT)}")
    return chunks

In [32]:
dispatcher = ParserDispatcher(parsers=[
    DoclingParser(chunk_size=cfg.CHUNK_SIZE, chunk_overlap=cfg.CHUNK_OVERLAP),
    StructuredDataParser(
        rows_per_chunk=1,
        chunk_size=cfg.CHUNK_SIZE,
        chunk_overlap=cfg.CHUNK_OVERLAP,
    ),
])

CACHE_PATH = cfg.DATA_PROCESSED_DIR / "phase0_chunks.json"

# Force re-parse by setting REPARSE = True
REPARSE = True

if not REPARSE and CACHE_PATH.exists():
    all_chunks = load_chunks_cache(CACHE_PATH)
else:
    all_chunks = dispatcher.parse_directory(cfg.DATA_RAW_DIR)
    if all_chunks:
        save_chunks_cache(all_chunks, CACHE_PATH)

# Quick peek: one sample chunk per format
if all_chunks:
    print("\nSample chunk per format:")
    by_fmt = sorted(all_chunks, key=lambda c: c.source_format)
    for fmt, group in groupby(by_fmt, key=lambda c: c.source_format):
        sample = next(group)
        print(f"\n--- {fmt} ---")
        print(f"  source : {sample.source_path}")
        print(f"  chars  : {len(sample.text)}")
        print(f"  text   : {sample.text[:220]!r}")
else:
    log.warning("No chunks produced — make sure data/raw/ contains supported files")

2026-05-27 11:44:48,166 - INFO    | rag | Found 74 supported file(s) under IPC/


Parsing:   0%|          | 0/74 [00:00<?, ?file/s]

2026-05-27 11:44:48,171 - INFO    | rag | [Docling] parsing IPC_OLD_split_0.pdf (pdf) …
2026-05-27 11:44:48,173 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-27 11:44:48,821 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
2026-05-27 11:44:49,868 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiScaleDeformableAttention/MultiScaleDeformableAttention.so: cannot open shared object file: No such file or directory
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiScaleDeformableAttention/MultiScaleDeformableAttention.so: cannot open shared object file: No such file or directory
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiScaleDef


Sample chunk per format:

--- pdf ---
  source : /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_0.pdf
  chars  : 108
  text   : 'INDIAN PENAL CODE, 1860 THE\n\nNO. 45 OF 1860 1* ACT\n\nOctober, 1860.] [6th\n\nI CHAPTER\n\nINTRODUCTION\n\nCHAPTER I'


In [33]:
from pathlib import Path
for p in dispatcher.parsers:
    print(f"{type(p).__name__}: supports .txt? {p.supports(Path('x.txt'))}")

DoclingParser: supports .txt? False
StructuredDataParser: supports .txt? True


In [34]:
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_ollama import OllamaEmbeddings


COLLECTION_NAME = "IPC_Corpus"

# 1) Embedding function — local, via Ollama
embeddings = OllamaEmbeddings(
    model=cfg.EMBEDDING_MODEL,
    base_url=cfg.OLLAMA_BASE_URL,
)

# Probe: confirm the model is reachable and capture dimension
_probe = embeddings.embed_query("hello world")
EMBED_DIM = len(_probe)
log.info(f"Embedding model '{cfg.EMBEDDING_MODEL}' OK → dim={EMBED_DIM}")


# 2) RagChunk → LangChain Document at the boundary
def chunks_to_documents(chunks: list[RagChunk]) -> tuple[list[Document], list[str]]:
    docs = [
        Document(page_content=c.text, metadata=c.to_langchain_metadata())
        for c in chunks
    ]
    ids = [c.chunk_id for c in chunks]
    return docs, ids


# 3) Open (or create) the persistent Chroma collection
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(cfg.CHROMA_PERSIST_DIR),
)

existing_count = vectorstore._collection.count()
log.info(f"Chroma '{COLLECTION_NAME}': {existing_count} vectors present")


# 4) Idempotent ingest — embed only NEW chunks
REINDEX = False  # flip to True to wipe and rebuild from scratch

if REINDEX and existing_count > 0:
    log.warning(f"REINDEX=True → wiping {existing_count} vectors")
    vectorstore.delete_collection()
    vectorstore = Chroma(
        collection_name=COLLECTION_NAME,
        embedding_function=embeddings,
        persist_directory=str(cfg.CHROMA_PERSIST_DIR),
    )
    existing_count = 0

if not all_chunks:
    log.warning("`all_chunks` is empty — run Piece 5 first, or load from cache")
else:
    docs, ids = chunks_to_documents(all_chunks)

    # Skip chunks already indexed (by chunk_id)
    if existing_count > 0:
        existing_ids = set(vectorstore.get(include=[])["ids"])
        to_add = [(d, i) for d, i in zip(docs, ids) if i not in existing_ids]
    else:
        to_add = list(zip(docs, ids))

    log.info(
        f"To embed: {len(to_add)} new ("
        f"{len(docs) - len(to_add)} already present)"
    )

    # Batch for progress visibility — Ollama on CPU can be slow at first
    BATCH_SIZE = 64
    for i in tqdm(range(0, len(to_add), BATCH_SIZE), desc="Embedding", unit="batch"):
        batch = to_add[i : i + BATCH_SIZE]
        if not batch:
            continue
        b_docs = [p[0] for p in batch]
        b_ids = [p[1] for p in batch]
        vectorstore.add_documents(documents=b_docs, ids=b_ids)

    final_count = vectorstore._collection.count()
    log.info(f"Chroma '{COLLECTION_NAME}': {final_count} vectors total")

2026-05-27 12:08:01,974 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:01,974 - INFO    | rag | Embedding model 'embeddinggemma:latest' OK → dim=768
2026-05-27 12:08:01,984 - INFO    | chromadb.telemetry.product.posthog | Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-05-27 12:08:07,083 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-05-27 12:08:07,357 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
2026-05-27 12:08:07,359 - INFO    | rag | Chroma 'IPC_Corpus': 0 vectors present
2026-05-27 12:08:07,366 - INFO    | rag | To embed: 1677 new (0 already present)


Embedding:   0%|          | 0/27 [00:00<?, ?batch/s]

2026-05-27 12:08:11,334 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:15,321 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:18,811 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:22,579 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:28,980 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:34,146 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:41,051 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:46,700 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:08:52,621 - INFO    | httpx | HTTP Request: POST http://localhost:11434/ap

In [35]:
def search_and_show(query: str, k: int | None = None) -> list[tuple[Document, float]]:
    k = k or cfg.TOP_K
    results = vectorstore.similarity_search_with_score(query, k=k)

    print(f"\n🔎 Query: {query!r}")
    print(f"Top {k} (Chroma cosine distance — lower = closer):\n")

    for rank, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc_bits = []
        if md.get("page_number"):   loc_bits.append(f"p.{md['page_number']}")
        if md.get("slide_number"):  loc_bits.append(f"slide {md['slide_number']}")
        if md.get("sheet_name"):    loc_bits.append(f"sheet '{md['sheet_name']}'")
        if md.get("row_range"):     loc_bits.append(f"rows {md['row_range']}")
        if md.get("section_title"): loc_bits.append(f"§ {md['section_title']}")
        loc = "  |  ".join(loc_bits)

        print(f"#{rank}  score={score:.4f}  [{md['source_format']}]  {md['source_path']}")
        if loc:
            print(f"     {loc}")
        preview = doc.page_content[:300].replace("\n", " ")
        print(f"     {preview!r}\n")

    return results


# Change this to something that actually appears in YOUR corpus
QUERY = "Child labor laws"
_ = search_and_show(QUERY)

2026-05-27 12:14:34,130 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:14:34,131 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



🔎 Query: 'Child labor laws'
Top 5 (Chroma cosine distance — lower = closer):

#1  score=0.9256  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf
     'Of wrongful restraint and wrongful confinement'

#2  score=0.9398  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf
     '## Of right of private defence'

#3  score=1.0536  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_19.pdf
     '## OF OFFENCES RELATING TO RELIGION'

#4  score=1.0595  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_21.pdf
     '## Of criminal trespass'

#5  score=1.0804  [pdf]  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf
     '## Of robbery and dacoity'



In [36]:
from langchain_ollama import ChatOllama
from langchain_core.messages import SystemMessage, HumanMessage

llm = ChatOllama(
    model=cfg.OLLAMA_MODEL,
    base_url=cfg.OLLAMA_BASE_URL,
    temperature=cfg.LLM_TEMPERATURE,
    num_ctx=cfg.LLM_NUM_CTX,
)

_smoke = llm.invoke("Reply with exactly: pong")
log.info(f"LLM '{cfg.OLLAMA_MODEL}' OK → {_smoke.content!r}")

2026-05-27 12:14:56,956 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:14:57,001 - INFO    | rag | LLM 'gemma-4-e4b:latest' OK → 'pong'


In [37]:
SYSTEM_PROMPT = """You are a precise assistant for company policy and internal documentation questions.

Rules:
1. Use ONLY the information in the CONTEXT below. Do not use outside knowledge.
2. If the answer is not in the context, reply exactly: "I don't have that information in the provided documents."
3. Cite every claim with the source tag in square brackets, e.g. [1] or [2, 3].
4. Quote short passages verbatim when they are decisive (e.g., policy clauses, IDs, dates).
5. Be concise. Do not pad or speculate."""


def _format_location(md: dict) -> str:
    bits = []
    if md.get("page_number"):   bits.append(f"p.{md['page_number']}")
    if md.get("slide_number"):  bits.append(f"slide {md['slide_number']}")
    if md.get("sheet_name"):    bits.append(f"sheet '{md['sheet_name']}'")
    if md.get("row_range"):     bits.append(f"rows {md['row_range']}")
    if md.get("section_title"): bits.append(f"§ {md['section_title']}")
    return " | ".join(bits)


def build_context(results: list[tuple[Document, float]]) -> tuple[str, list[dict]]:
    """Format retrieved chunks as numbered context + return a citation table."""
    blocks: list[str] = []
    citations: list[dict] = []
    for i, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc = _format_location(md)
        header = f"[{i}] {md['source_path']}" + (f" | {loc}" if loc else "")
        blocks.append(f"{header}\n{doc.page_content}")
        citations.append({
            "tag": i,
            "source_path": md["source_path"],
            "source_format": md["source_format"],
            "location": loc,
            "score": float(score),
            "chunk_id": md.get("chunk_id"),
        })
    return "\n\n---\n\n".join(blocks), citations


def answer(query: str, k: int | None = None, show_context: bool = False) -> dict:
    """End-to-end RAG: retrieve → build prompt → generate → return structured result."""
    k = k or cfg.TOP_K
    results = vectorstore.similarity_search_with_score(query, k=k)

    if not results:
        return {
            "question": query,
            "answer": "I don't have any relevant documents indexed.",
            "citations": [],
        }

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"

    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])

    out = {
        "question": query,
        "answer": response.content,
        "citations": citations,
    }
    if show_context:
        out["context"] = context_block
    return out


def pretty_print(result: dict) -> None:
    print(f"{result['question']}\n")
    print(f"{result['answer']}\n")
    if result["citations"]:
        print("Sources:")
        for c in result["citations"]:
            loc = f"  |  {c['location']}" if c["location"] else ""
            print(f"   [{c['tag']}] {c['source_path']}{loc}   (dist={c['score']:.3f})")

In [41]:
# Try a query that should be answerable from YOUR corpus
QUERY = "Child safety"

result = answer(QUERY)
pretty_print(result)

# Sanity check the model honours "I don't know"
print("\n" + "=" * 60 + "\n")
nonsense = answer("what's the recipe for chocolate cake")
pretty_print(nonsense)

2026-05-27 12:17:01,403 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:17:01,789 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:17:18,990 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Child safety

I don't have that information in the provided documents.

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf   (dist=0.959)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_12.pdf   (dist=0.995)
   [3] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf   (dist=1.002)
   [4] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf   (dist=1.038)
   [5] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_13.pdf   (dist=1.039)




2026-05-27 12:17:19,237 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


what's the recipe for chocolate cake

I don't have that information in the provided documents.

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf   (dist=1.198)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_18.pdf   (dist=1.244)
   [3] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_12.pdf   (dist=1.285)
   [4] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_19.pdf   (dist=1.286)
   [5] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_16.pdf   (dist=1.293)


In [ ]:
# [Files in data/raw/]
#         ↓  ParserDispatcher
#    [RagChunk[]]
#         ↓  OllamaEmbeddings (embeddinggemma)
#    [Chroma index]
#         ↓  similarity_search_with_score
#    [Top-k chunks]
#         ↓  build_context → SYSTEM_PROMPT
#    [Gemma (gemma-4-e4b)]
#         ↓
#    [Answer + citations]

: 

In [ ]:
# [████████████░░░░░░░░░░░░░░░░░░░░░░] ~35%

# ✓ PHASE 0 — Naive RAG     (DONE — your baseline)
#   → PHASE 1 — Hybrid Retrieval ◄── NEXT
#     PHASE 2 — Query Intelligence
#     PHASE 3 — Agentic Orchestration
#     PHASE 4 — Evaluation & Observability
#     PHASE 5 — Production Hardening
#     PHASE 6 — Cloud & Scale

In [ ]:
# phase 1
# hybrid retrieval

In [42]:
import re
from rank_bm25 import BM25Okapi


class BM25Retriever:
    """In-memory BM25 sparse retriever over RagChunks.

    Returns LangChain Documents so it composes with the dense retriever
    under the same interface. Score is BM25 (higher = better) — opposite
    of Chroma distance (lower = better). RRF fusion in the next piece
    sidesteps this by using rank, not raw score.
    """

    # Keep hyphens (vendor IDs like V-001), drop other punctuation.
    _TOKEN_RE = re.compile(r"[^\w\s\-]")

    def __init__(self, chunks: list[RagChunk], min_token_len: int = 2):
        if not chunks:
            raise ValueError("BM25Retriever needs at least one chunk")
        self.chunks = chunks
        self.min_token_len = min_token_len

        log.info(f"[BM25] tokenizing {len(chunks)} chunks …")
        self._tokenized_corpus = [self._tokenize(c.text) for c in chunks]
        self.bm25 = BM25Okapi(self._tokenized_corpus)

        # Pre-build LangChain Documents so retrieval is just a lookup
        self._docs = [
            Document(page_content=c.text, metadata=c.to_langchain_metadata())
            for c in chunks
        ]
        avg_tokens = sum(len(t) for t in self._tokenized_corpus) / len(chunks)
        log.info(f"[BM25] index ready | {len(chunks)} docs | avg {avg_tokens:.0f} tokens/doc")

    def _tokenize(self, text: str) -> list[str]:
        text = text.lower()
        text = self._TOKEN_RE.sub(" ", text)
        return [t for t in text.split() if len(t) >= self.min_token_len]

    def retrieve(
        self,
        query: str,
        k: int = 10,
        min_score: float = 0.0,
    ) -> list[tuple[Document, float]]:
        tokens = self._tokenize(query)
        if not tokens:
            return []
        scores = self.bm25.get_scores(tokens)
        # Argsort top-k
        ranked_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
        return [
            (self._docs[i], float(scores[i]))
            for i in ranked_idx
            if scores[i] > min_score
        ]



In [43]:

# Build the index from chunks already in memory (or load from cache first if needed)
if not all_chunks:
    all_chunks = load_chunks_cache(cfg.DATA_PROCESSED_DIR / "phase0_chunks.json")

bm25_retriever = BM25Retriever(all_chunks)

2026-05-27 12:17:42,686 - INFO    | rag | [BM25] tokenizing 1677 chunks …
2026-05-27 12:17:42,734 - INFO    | rag | [BM25] index ready | 1677 docs | avg 82 tokens/doc


In [44]:
def show_results(title: str, results: list[tuple[Document, float]], score_label: str) -> None:
    print(f"\n--- {title} ---")
    if not results:
        print("  (no results)")
        return
    for rank, (doc, score) in enumerate(results, 1):
        md = doc.metadata
        loc = _format_location(md)
        loc_str = f"  |  {loc}" if loc else ""
        preview = doc.page_content[:120].replace("\n", " ")
        print(f"  #{rank}  {score_label}={score:.3f}  {md['source_path']}{loc_str}")
        print(f"        {preview!r}")


# Try BOTH a literal/keyword query and a semantic/paraphrase query
TEST_QUERIES = [
    "Federal agencies are in possession of documents pertaining to gross human rights violations abroad which are needed by foreign authorities to document",                              # literal ID — BM25 should win
    "Human Rights Information Act",        # exact-ish phrase — both should hit
    "Girls Count Act of 2014",   # paraphrase — dense should win
]

for q in TEST_QUERIES:
    print("=" * 70)
    print(f"🔎 Query: {q!r}")

    dense_results = vectorstore.similarity_search_with_score(q, k=5)
    bm25_results = bm25_retriever.retrieve(q, k=5)

    show_results("DENSE (Chroma, embeddinggemma)", dense_results, "dist")
    show_results("BM25  (rank_bm25)",              bm25_results,  "bm25")

2026-05-27 12:17:53,468 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:17:53,543 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


🔎 Query: 'Federal agencies are in possession of documents pertaining to gross human rights violations abroad which are needed by foreign authorities to document'

--- DENSE (Chroma, embeddinggemma) ---
  #1  dist=1.182  /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_45.pdf
        '. documents material.--Whoever material, any such appearance to possession device *[imprisonment for to'
  #2  dist=1.252  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf
        '## Of fraudulent deeds and dispositions of property'
  #3  dist=1.264  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf
        'Of wrongful restraint and wrongful confinement'
  #4  dist=1.289  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf
        '## Of receiving stolen property'
  #5  dist=1.311  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_4.pdf
        '## Of criminal misappropriation of property'

--- BM25  (rank_bm25) ---
  #1  bm25=23.482  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_4.

2026-05-27 12:17:53,614 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"



--- DENSE (Chroma, embeddinggemma) ---
  #1  dist=1.062  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_2.pdf
        '## CHAPTER XIV  OF FALSE EVIDENCE AND OFFENCES AGAINST PUBLIC JUSTICE'
  #2  dist=1.195  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf
        '## Of right of private defence'
  #3  dist=1.230  /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_13.pdf
        'all  lawful means in his or their to prevent it and, in the event of its  taking place, do not use lawful  means  in  hi'
  #4  dist=1.249  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_20.pdf
        '## Of receiving stolen property'
  #5  dist=1.268  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_10.pdf
        '## OF OFFENCES AGAINST WOMAN AND CHILD  Of sexual offences'

--- BM25  (rank_bm25) ---
  #1  bm25=5.506  /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_1.pdf
        '- 14. Act done by a person bound, or by mistake of fact believing himself bound, by law. - 15. Act of Judge when act

In [45]:
## combining both retrievers: 
## RRF_score (doc) = Σ over retrievers:  weight / (k + rank_of_doc_in_that_retriever)

In [46]:
def reciprocal_rank_fusion(
    rankings: list[list[Document]],
    k: int = 60,
    weights: list[float] | None = None,
) -> list[tuple[Document, float]]:
    """Fuse multiple ranked Document lists via Reciprocal Rank Fusion.

    Uses rank position only — immune to score-scale mismatches between
    retrievers. Documents are deduped by chunk_id from their metadata.
    """
    if weights is None:
        weights = [1.0] * len(rankings)
    if len(weights) != len(rankings):
        raise ValueError("weights must match number of rankings")

    scores: dict[str, float] = defaultdict(float)
    doc_lookup: dict[str, Document] = {}

    for ranking, w in zip(rankings, weights):
        for rank, doc in enumerate(ranking, start=1):
            doc_id = doc.metadata.get("chunk_id") or doc.metadata.get("content_hash")
            if not doc_id:
                continue
            scores[doc_id] += w / (k + rank)
            doc_lookup.setdefault(doc_id, doc)

    return sorted(
        ((doc_lookup[did], s) for did, s in scores.items()),
        key=lambda x: x[1],
        reverse=True,
    )


class EnsembleRetriever:
    """Hybrid retriever = dense ⊕ BM25, fused via RRF.

    fetch_k: how many to pull from EACH base retriever before fusion, default 20.
    top_k:   how many to return after fusion, default 5.
    rrf_k:   the "k" parameter for RRF fusion, default 60 (per the research paper's recommendation).

    Weights are per-retriever. Equal weights (1.0, 1.0) is a strong default.
    Bump dense weight up if your corpus is paraphrase-heavy; bump BM25 up
    if it's full of IDs / codes / exact references.
    """

    def __init__(
        self,
        dense_store: Chroma,
        sparse_retriever: BM25Retriever,
        fetch_k: int = 20, 
        weights: tuple[float, float] = (1.0, 1.0),
        rrf_k: int = 60,
    ):
        self.dense_store = dense_store
        self.sparse_retriever = sparse_retriever
        self.fetch_k = fetch_k
        self.weights = list(weights)
        self.rrf_k = rrf_k

    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Document, float]]:
        # Dense: we only need order, not the distance, for RRF
        dense_docs = self.dense_store.similarity_search(query, k=self.fetch_k)

        # Sparse: returns (doc, bm25_score); keep just docs, in rank order
        sparse_pairs = self.sparse_retriever.retrieve(query, k=self.fetch_k)
        sparse_docs = [d for d, _ in sparse_pairs]

        fused = reciprocal_rank_fusion(
            rankings=[dense_docs, sparse_docs],
            k=self.rrf_k,
            weights=self.weights,
        )
        return fused[:top_k]




In [47]:
ensemble = EnsembleRetriever(
    dense_store=vectorstore,
    sparse_retriever=bm25_retriever,
    fetch_k=20,
    weights=(1.0, 1.0),
)
log.info(f"[Ensemble] ready | fetch_k={ensemble.fetch_k} | weights={ensemble.weights}")

2026-05-27 12:18:18,486 - INFO    | rag | [Ensemble] ready | fetch_k=20 | weights=[1.0, 1.0]


In [48]:
## comparing dense vs. bm25 vs hybrid

In [52]:
# Redefine pretty_print to accept a score-label (replaces the Phase 0 version)
def pretty_print(result: dict, score_label: str = "score") -> None:
    print(f"{result['question']}\n")
    print(f"{result['answer']}\n")
    if result["citations"]:
        print("Sources:")
        for c in result["citations"]:
            loc = f"  |  {c['location']}" if c["location"] else ""
            print(f"   [{c['tag']}] {c['source_path']}{loc}   ({score_label}={c['score']:.3f})")


def answer_phase1(
    query: str,
    k: int | None = None,
    retriever: str = "hybrid",  # "dense" | "bm25" | "hybrid"
) -> dict:
    k = k or cfg.TOP_K

    if retriever == "dense":
        results = vectorstore.similarity_search_with_score(query, k=k)
    elif retriever == "bm25":
        results = bm25_retriever.retrieve(query, k=k)
    elif retriever == "hybrid":
        results = ensemble.retrieve(query, top_k=k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if not results:
        return {"question": query, "answer": "No relevant docs.", "citations": [], "retriever": retriever}

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])
    return {
        "question": query,
        "answer": response.content,
        "citations": citations,
        "retriever": retriever,
    }


SCORE_LABELS = {"dense": "dist", "bm25": "bm25", "hybrid": "rrf"}

# Pick ONE query from your corpus and watch how the three retrievers differ
TEST_QUERY = "Right of private defence against the act of a person of unsound mind"

for r in ("dense", "bm25", "hybrid"):
    print("=" * 72)
    print(f"RETRIEVER: {r.upper()}")
    print("=" * 72)
    pretty_print(answer_phase1(TEST_QUERY, retriever=r), score_label=SCORE_LABELS[r])
    print()

2026-05-27 12:22:09,172 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


RETRIEVER: DENSE


2026-05-27 12:22:09,884 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Right of private defence against the act of a person of unsound mind

When an act that would otherwise be an offense is committed by a person due to factors such as "the youth, the want of maturity of understanding, the unsoundness of mind or the intoxication of the person doing that act, or by reason of any misconception on the part of that person," every person retains the same right of private defence against that act as if the act were the actual offense [2].

Illustrations provided include:

*   **Unsound Mind:** If Z, a person of unsound mind, attempts to kill A, Z is guilty of no offense, but A has the same right of private defence which he would have if Z were sane [5].
*   **Misconception:** If Z attacks A because Z mistakenly believes A is a house-breaker, Z commits no offense, but A has the same right of private defence against Z, which he would have if Z were not acting under that misconception [5].

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf   (

2026-05-27 12:22:56,118 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:23:34,106 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Right of private defence against the act of a person of unsound mind

When an act that would otherwise be a certain offence is committed by a person due to "the youth, the want of maturity of understanding, the unsoundness of mind or the intoxication of the person doing that act, or by reason of any misconception on the part of that person," every person has the same right of private defence against that act which they would have if the act were a full offence [2].

For example, if "Z, a person of unsound mind, attempts to kill A; Z is guilty of no offence. But A has the same right of private defence which he would have if Z were sane" [4].

This right is covered under the section titled "Right of private defence against act of a person of unsound mind, etc." [3, 2].

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/IPC/IPC_OLD_split_6.pdf   (bm25=47.796)
   [2] /home/thimu/Downloads/pdf_splitter/IPC/IPC_split_7.pdf   (bm25=43.230)
   [3] /home/thimu/Downloads/pdf_splitter/IPC/IPC_sp

2026-05-27 12:23:34,753 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Right of private defence against the act of a person of unsound mind

When an act would otherwise constitute an offense, but is committed by a person who is:
*   Young;
*   Lacking maturity of understanding;
*   Unsound of mind;
*   Intoxicated; or
*   Acting due to a misconception [2];

the person defending themselves has the same right of private defence against that act as they would if the act were fully criminal [1, 2].

Examples of this right include:
*   If Z, a person of unsound mind, attempts to kill A, Z is guilty of no offense, but A retains the same right of private defence as if Z were sane [3].
*   If Z attacks A under a misconception (e.g., mistaking A for a house-breaker), Z commits no offense, but A has the same right of private defence against Z as if Z were not acting under that misconception [3].

This topic is covered under Section 36, "Right of private defence against act of a person of unsound mind, etc." [4].

Sources:
   [1] /home/thimu/Downloads/pdf_splitter/I

In [ ]:
## reranker: pulling from top 20 retrieved chuncks

In [53]:
from FlagEmbedding import FlagReranker


class Reranker:
    """Thin wrapper around BGE cross-encoder rerankers."""

    def __init__(
        self,
        model_name: str = "BAAI/bge-reranker-base",
        use_fp16: bool = True,
        normalize: bool = True,
    ):
        log.info(f"[Reranker] loading {model_name} (first call downloads ~1GB)…")
        self.model = FlagReranker(model_name, use_fp16=use_fp16)
        self.model_name = model_name
        self.normalize = normalize  # sigmoid → 0..1 scores, intuitive thresholds
        log.info(f"[Reranker] ready")

    def score(self, query: str, docs: list[Document]) -> list[float]:
        if not docs:
            return []
        pairs = [[query, d.page_content] for d in docs]
        out = self.model.compute_score(pairs, normalize=self.normalize)
        # FlagReranker returns float for a single pair, list for multiple
        if isinstance(out, float):
            return [out]
        return [float(x) for x in out]


class RerankedRetriever:
    """Wrap any base retriever; over-fetch then rerank with a cross-encoder.

    Pipeline:  base.retrieve(top_k=fetch_k)  →  rerank  →  top_k

    Cuts the noise that hybrid retrieval inevitably brings in.
    """

    def __init__(
        self,
        base_retriever,
        reranker: Reranker,
        fetch_k: int = 20,
        min_score: float | None = None,  # set in Phase 4 eval, not by vibes
    ):
        self.base = base_retriever
        self.reranker = reranker
        self.fetch_k = fetch_k
        self.min_score = min_score

    def retrieve(self, query: str, top_k: int = 5) -> list[tuple[Document, float]]:
        candidates = self.base.retrieve(query, top_k=self.fetch_k)
        if not candidates:
            return []

        docs = [d for d, _ in candidates]
        scores = self.reranker.score(query, docs)

        scored = sorted(zip(docs, scores), key=lambda x: x[1], reverse=True)
        if self.min_score is not None:
            scored = [(d, s) for d, s in scored if s >= self.min_score]
        return scored[:top_k]


In [54]:
# download the reranker model
reranker = Reranker(model_name="BAAI/bge-reranker-base", use_fp16=True, normalize=True)

hybrid_reranked = RerankedRetriever(
    base_retriever=ensemble,    # our hybrid retriever from Cell 19
    reranker=reranker,
    fetch_k=20,
    min_score=None,             # learn this from eval, not intuition
)
log.info(f"[Pipeline] hybrid+rerank ready | fetch_k={hybrid_reranked.fetch_k}")

2026-05-27 12:25:53,874 - INFO    | rag | [Reranker] loading BAAI/bge-reranker-base (first call downloads ~1GB)…
2026-05-27 12:25:55,079 - INFO    | rag | [Reranker] ready
2026-05-27 12:25:55,080 - INFO    | rag | [Pipeline] hybrid+rerank ready | fetch_k=20


In [55]:
## testing hybrid vs hybrid+rerank on the same query

In [56]:
def answer_phase1(
    query: str,
    k: int | None = None,
    retriever: str = "hybrid_reranked",   # new default
) -> dict:
    k = k or cfg.TOP_K

    if retriever == "dense":
        results = vectorstore.similarity_search_with_score(query, k=k)
    elif retriever == "bm25":
        results = bm25_retriever.retrieve(query, k=k)
    elif retriever == "hybrid":
        results = ensemble.retrieve(query, top_k=k)
    elif retriever == "hybrid_reranked":
        results = hybrid_reranked.retrieve(query, top_k=k)
    else:
        raise ValueError(f"Unknown retriever: {retriever}")

    if not results:
        return {"question": query, "answer": "No relevant docs.", "citations": [], "retriever": retriever}

    context_block, citations = build_context(results)
    user_message = f"CONTEXT:\n{context_block}\n\nQUESTION: {query}\n\nANSWER:"
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_message),
    ])
    return {
        "question": query,
        "answer": response.content,
        "citations": citations,
        "retriever": retriever,
    }


SCORE_LABELS = {
    "dense": "dist", "bm25": "bm25", "hybrid": "rrf", "hybrid_reranked": "rerank",
}

# Use a query where the right answer needs precision, not just recall
TEST_QUERY = "Right of private defence"

for r in ("hybrid", "hybrid_reranked"):
    print("=" * 72)
    print(f"RETRIEVER: {r.upper()}")
    print("=" * 72)
    pretty_print(answer_phase1(TEST_QUERY, retriever=r), score_label=SCORE_LABELS[r])
    print()

2026-05-27 12:26:24,859 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


RETRIEVER: HYBRID


2026-05-27 12:26:25,567 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:27:14,718 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"


Right of private defence

The right of private defence is governed by several sections of the law:

**General Principles and Scope**

*   **No Offence:** Nothing is considered an offence if it is done while exercising the right of private defence [3].
*   **Right to Defend:** Every person has a right, subject to restrictions in Section 37, to defend:
    *   **Body:** "his own body, and the body of any other person, against any offence affecting the human body" [3].
    *   **Property:** "the property, whether movable or immovable, of himself or of any other person, against any act which is an offence falling under the definition of theft, robbery, mischief or criminal trespass, or which is an attempt to commit theft, robbery, mischief or criminal trespass" [3].

**Extensions and Specific Rules**

*   **Property Defence Extending to Death:** The right of private defence of property extends, under the restrictions specified in Section 37, to "the voluntary causing of death or of any oth

You're using a XLMRobertaTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.
2026-05-27 12:27:15,797 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"


Right of private defence

The right of private defence covers several aspects, including protection against acts by persons of unsound mind, and specific rules regarding the body and property.

**Against Persons of Unsound Mind:**
If an act that would normally be an offense is not so because the person committing it is due to "the youth, the want of maturity of understanding, the unsoundness of mind or the intoxication of the person doing that act, or by reason of any misconception on the part of that person," the person retains the same right of private defence as if the act were the actual offense [2].

**Right of Private Defence of the Body:**
*   **Commencement and Continuance:** The right of private defence of the body begins "as soon as a reasonable apprehension of danger to the body arises from an attempt or threat to commit the offence, though the offence may not have been committed" [5]. It continues "as long as such apprehension of danger to the body continues" [5].
*   **Har

In [57]:
#pip install "docling-core[chunking]" --break-system-packages

## phase 1: hybrid chunker

In [58]:
from docling.chunking import HybridChunker
from docling_core.transforms.chunker.tokenizer.huggingface import HuggingFaceTokenizer
from transformers import AutoTokenizer


class DoclingHybridParser:
    """Phase 1: structure-aware chunking with real provenance.

    vs DoclingParser (Phase 0):
      • chunks respect headings/tables/sections (no mid-table slicing)
      • real page_number (PDF) / slide_number (PPTX)
      • section hierarchy captured in section_title
      • heading context prepended to text → better embeddings
    """

    SUPPORTED: dict[str, SourceFormat] = {
        ".pdf": "pdf", ".pptx": "pptx", ".docx": "docx",
        ".html": "html", ".htm": "html", ".md": "md",
    }

    def __init__(
        self,
        max_tokens: int = 512,
        tokenizer_id: str = "sentence-transformers/all-MiniLM-L6-v2",
        merge_peers: bool = True,
    ):
        self.converter = DocumentConverter()
        tok = HuggingFaceTokenizer(
            tokenizer=AutoTokenizer.from_pretrained(tokenizer_id),
            max_tokens=max_tokens,
        )
        self.chunker = HybridChunker(tokenizer=tok, merge_peers=merge_peers)

    def supports(self, path: Path) -> bool:
        return path.suffix.lower() in self.SUPPORTED

    def parse(self, path: Path) -> list[RagChunk]:
        if not self.supports(path):
            raise ValueError(f"DoclingHybridParser does not support {path.suffix}")

        fmt = self.SUPPORTED[path.suffix.lower()]
        log.info(f"[DoclingHybrid] parsing {path.name} ({fmt}) …")
        try:
            result = self.converter.convert(str(path))
        except Exception as e:
            log.error(f"[DoclingHybrid] convert failed for {path.name}: {e}")
            return []

        doc = result.document
        rel = self._relative_source(path)
        chunks: list[RagChunk] = []

        for dl_chunk in self.chunker.chunk(dl_doc=doc):
            # contextualize() prepends the heading hierarchy → richer embeddings
            text = self.chunker.contextualize(chunk=dl_chunk)
            if not text or not text.strip():
                continue

            page_no = self._first_page_no(dl_chunk)
            chunks.append(RagChunk(
                text=text,
                source_path=rel,
                source_format=fmt,
                page_number=page_no if fmt == "pdf" else None,
                slide_number=page_no if fmt == "pptx" else None,
                section_title=self._section_title(dl_chunk),
                element_type=self._element_type(dl_chunk),
            ))

        log.info(f"[DoclingHybrid] {path.name}: {len(chunks)} structure-aware chunks")
        return chunks

    @staticmethod
    def _first_page_no(dl_chunk) -> int | None:
        try:
            for item in dl_chunk.meta.doc_items:
                for prov in getattr(item, "prov", []) or []:
                    pn = getattr(prov, "page_no", None)
                    if pn is not None:
                        return int(pn)
        except Exception:
            pass
        return None

    @staticmethod
    def _section_title(dl_chunk) -> str | None:
        try:
            headings = getattr(dl_chunk.meta, "headings", None)
            if headings:
                return " > ".join(h for h in headings if h)[:300]
        except Exception:
            pass
        return None

    @staticmethod
    def _element_type(dl_chunk) -> ElementType:
        try:
            for item in dl_chunk.meta.doc_items:
                label = str(getattr(item, "label", "")).lower()
                if "table" in label: return "table"
                if "list" in label:  return "list"
                if "code" in label:  return "code"
        except Exception:
            pass
        return "text"

    @staticmethod
    def _relative_source(path: Path) -> str:
        try:
            return str(path.relative_to(PROJECT_ROOT))
        except ValueError:
            return str(path)

In [59]:
## re-parse + cheap peek
dispatcher_v2 = ParserDispatcher(parsers=[
    DoclingHybridParser(max_tokens=512, merge_peers=True),
    StructuredDataParser(
        rows_per_chunk=1,
        chunk_size=cfg.CHUNK_SIZE,
        chunk_overlap=cfg.CHUNK_OVERLAP,
    ),
])

all_chunks_v2 = dispatcher_v2.parse_directory(cfg.DATA_RAW_DIR)
save_chunks_cache(all_chunks_v2, cfg.DATA_PROCESSED_DIR / "phase1_chunks.json")

# Confirm page_number + section_title are now populated on a PDF chunk
pdf_chunks = [c for c in all_chunks_v2 if c.source_format == "pdf"]
if pdf_chunks:
    sample = next((c for c in pdf_chunks if c.page_number is not None), pdf_chunks[0])
    print("Structure-aware PDF chunk:")
    print(f"  page_number   : {sample.page_number}")
    print(f"  section_title : {sample.section_title}")
    print(f"  element_type  : {sample.element_type}")
    print(f"  text preview  : {sample.text[:320]!r}")
else:
    log.warning("No PDF chunks — add a PDF to data/raw/pdfs/ to see page numbers")

/home/thimu/github_vs/protoRAG/rag-pipeline/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
2026-05-27 12:28:36,140 - INFO    | rag | Found 74 supported file(s) under IPC/


Parsing:   0%|          | 0/74 [00:00<?, ?file/s]

2026-05-27 12:28:36,143 - INFO    | rag | [DoclingHybrid] parsing IPC_OLD_split_0.pdf (pdf) …
2026-05-27 12:28:36,145 - INFO    | docling.document_converter | Going to convert document batch...
2026-05-27 12:28:36,731 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
2026-05-27 12:28:37,734 - INFO    | docling.utils.accelerator_utils | Accelerator device: 'cuda:0'
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiScaleDeformableAttention/MultiScaleDeformableAttention.so: cannot open shared object file: No such file or directory
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiScaleDeformableAttention/MultiScaleDeformableAttention.so: cannot open shared object file: No such file or directory
Could not load the custom kernel for multi-scale deformable attention: /home/thimu/.cache/torch_extensions/py312_cu130/MultiSc

Structure-aware PDF chunk:
  page_number   : 1
  section_title : None
  element_type  : text
  text preview  : 'INDIAN PENAL CODE, 1860 THE\nNO. 45 OF 1860 1* ACT\nOctober, 1860.] [6th\nI CHAPTER\nINTRODUCTION\nCHAPTER I'


In [60]:
all_chunks = all_chunks_v2  # promote to the working corpus

# 1) Chroma: chunk boundaries changed → full re-embed required
log.warning("Rebuilding Chroma with structure-aware chunks (one-time re-embed)…")
vectorstore.delete_collection()
vectorstore = Chroma(
    collection_name=COLLECTION_NAME,
    embedding_function=embeddings,
    persist_directory=str(cfg.CHROMA_PERSIST_DIR),
)
docs, ids = chunks_to_documents(all_chunks)
BATCH = 64
for i in tqdm(range(0, len(docs), BATCH), desc="Re-embedding", unit="batch"):
    vectorstore.add_documents(documents=docs[i:i + BATCH], ids=ids[i:i + BATCH])
log.info(f"Chroma rebuilt → {vectorstore._collection.count()} vectors")

# 2) BM25: rebuild from new chunks
bm25_retriever = BM25Retriever(all_chunks)

# 3) Rewire ensemble + reranker (they held refs to the OLD objects)
ensemble = EnsembleRetriever(vectorstore, bm25_retriever, fetch_k=20, weights=(1.0, 1.0))
hybrid_reranked = RerankedRetriever(ensemble, reranker, fetch_k=20)
log.info("Phase 1 pipeline rebuilt — structure-aware end to end")

# 4) Verify: citations should now show p.N and § Section
result = answer_phase1("Right of private defence against the act of a person of unsound mind",
                        retriever="hybrid_reranked")
pretty_print(result, score_label="rerank")

2026-05-27 12:30:35,594 - WARNING | rag | Rebuilding Chroma with structure-aware chunks (one-time re-embed)…
2026-05-27 12:30:36,168 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
2026-05-27 12:30:37,756 - ERROR   | chromadb.telemetry.product.posthog | Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Re-embedding:   0%|          | 0/10 [00:00<?, ?batch/s]

2026-05-27 12:30:55,395 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:31:19,038 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:31:43,489 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:32:08,272 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:32:29,256 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:32:45,554 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:33:12,107 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:33:32,076 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 12:33:52,120 - INFO    | httpx | HTTP Request: POST http://localhost:11434/ap

Right of private defence against the act of a person of unsound mind

When an act would otherwise be a certain offence, but is committed by a person due to factors such as youth, lack of maturity of understanding, unsoundness of mind, or intoxication, or due to a misconception on their part, every person retains the same right of private defence against that act as they would if the act were the actual offence [3, 5].

Specifically, the right of private defence against the act of a person of unsound mind, etc., applies when:
*   The act is not the offence due to the "youth, the want of maturity of understanding, the unsoundness of mind or the intoxication of the person doing that act, or by reason of any misconception on the part of that person" [3, 5].
*   In such cases, "every person has the same right of private defence against that act which he would have if the act were that offence" [3, 5].

For example, if a person of unsound mind attempts to kill another person, that person is 

In [ ]:
# [████████████████████░░░░░░░░░░░░░░] ~60%

# ✓ PHASE 0 — Naive RAG            (baseline)
# ✓ PHASE 1 — Hybrid Retrieval     (DONE)
#     ✓ BM25 sparse retriever
#     ✓ Ensemble (dense ⊕ BM25, RRF fusion)
#     ✓ Cross-encoder reranker (BGE)
#     ✓ Structure-aware chunking + real page/section citations
#   → next: PHASE 2 or PHASE 4

## phase 1 and phase 2 done
## moving to phase 4 evaluation & metrics

## adding query intelligence in the next step, once the evaluation framework is in place to show its impact

In [ ]:
# import random


# class EvalExample(BaseModel):
#     """One labelled eval case. Source-path level → survives re-chunking.

#     Convention: gold_source_paths == []  means  "system SHOULD refuse / find nothing"
#     (used later to test the 'I don't know' path and the min_score threshold).
#     """
#     question: str
#     gold_source_paths: list[str]
#     gold_snippets: list[str] = Field(default_factory=list)
#     reference_answer: str | None = None
#     difficulty: Literal["easy", "medium", "hard"] = "medium"
#     notes: str = ""


# QGEN_PROMPT = """You are creating evaluation questions for an Indian Penal Code knowledge base.

# Given the SOURCE PASSAGE below, write ONE realistic question a lawyer, student, or legal researcher would ask, where THIS passage contains the answer.

# Rules:
# - Answerable using ONLY facts in this passage.
# - Do NOT reference "this document/passage/section/above". Ask as if you don't know where the answer lives.
# - Prefer specific factual questions (section numbers, defined terms, punishments, illustrations) over vague ones.
# - Also copy the SHORT exact answer snippet verbatim from the passage.

# Return STRICT JSON, no markdown fences:
# {{"question": "...", "answer_snippet": "...", "difficulty": "easy|medium|hard"}}

# SOURCE PASSAGE:
# {passage}"""


# def generate_eval_set(
#     chunks: list[RagChunk],
#     n: int = 20,
#     seed: int = 42,
#     min_chunk_chars: int = 200,
# ) -> list[EvalExample]:
#     """LLM-synthesised eval set. CURATE the output by hand afterwards — not optional."""
#     rng = random.Random(seed)
#     pool = [c for c in chunks if len(c.text) >= min_chunk_chars] or list(chunks)
#     sample = rng.sample(pool, min(n, len(pool)))

#     examples: list[EvalExample] = []
#     for i, ch in enumerate(tqdm(sample, desc="Generating Q", unit="q"), 1):
#         prompt = QGEN_PROMPT.format(passage=ch.text[:2000])
#         try:
#             resp = llm.invoke([HumanMessage(content=prompt)])
#             raw = re.sub(r"^```(?:json)?|```$", "", resp.content.strip(),
#                          flags=re.MULTILINE).strip()
#             data = json.loads(raw)
#         except Exception as e:
#             log.warning(f"Q{i}: generation/parse failed ({e}); skipping")
#             continue

#         q = (data.get("question") or "").strip()
#         snippet = (data.get("answer_snippet") or "").strip()
#         diff = data.get("difficulty", "medium")
#         diff = diff if diff in ("easy", "medium", "hard") else "medium"

#         if len(q) < 12 or any(bad in q.lower() for bad in (
#             "this document", "this passage", "the table", "above", "the text", "this section"
#         )):
#             log.warning(f"Q{i}: degenerate question rejected: {q!r}")
#             continue

#         examples.append(EvalExample(
#             question=q,
#             gold_source_paths=[ch.source_path],
#             gold_snippets=[snippet] if snippet else [],
#             difficulty=diff,
#             notes=f"auto-gen from {ch.source_format} chunk {ch.chunk_id[:8]}",
#         ))

#     log.info(f"Generated {len(examples)} eval examples (requested {n})")
#     return examples


# def save_eval_set(examples: list[EvalExample], path: Path) -> None:
#     path.parent.mkdir(parents=True, exist_ok=True)
#     path.write_text(
#         json.dumps([e.model_dump() for e in examples], indent=2, ensure_ascii=False),
#         encoding="utf-8",
#     )
#     log.info(f"Saved {len(examples)} examples → {path.relative_to(PROJECT_ROOT)}")


# def load_eval_set(path: Path) -> list[EvalExample]:
#     if not path.exists():
#         return []
#     return [EvalExample(**d) for d in json.loads(path.read_text(encoding="utf-8"))]

### generating QnA for evaluation

In [62]:
import random
import time


EASY_PROMPT = """Create an EASY question for an Indian Penal Code knowledge base.

Direct factual lookup answerable by 1-10 words copied verbatim from the passage.
Use similar vocabulary to the passage. Ask about: section number, defined term, punishment, name, date.

Return STRICT JSON, no markdown:
{{"question": "...", "answer_snippet": "...", "reference_answer": "..."}}

PASSAGE:
{passage}"""


MEDIUM_PROMPT = """Create a MEDIUM-difficulty question for an Indian Penal Code knowledge base.

Rules:
- Paraphrase the legal terms (e.g. "stealing" not "theft", "intent to harm" not "mens rea").
- NOT answerable by keyword matching.
- 1-3 sentence answer derived from the passage.

Return STRICT JSON, no markdown:
{{"question": "...", "answer_snippet": "...", "reference_answer": "..."}}

PASSAGE:
{passage}"""


HARD_PROMPT = """Create a HARD question for an Indian Penal Code knowledge base.

You have TWO passages. Write ONE question requiring BOTH to answer.
Pick one of: COMPARISON, APPLICATION (hypothetical scenario), EXCEPTION REASONING, CROSS-REFERENCE.

Rules:
- Paraphrased legal vocabulary — do NOT echo the passages.
- Answer (2-4 sentences) MUST require facts from both passages.
- Do NOT reference "passage A/B" in the question.

Return STRICT JSON, no markdown:
{{"question": "...", "answer_snippet": "...", "reference_answer": "..."}}

PASSAGE A:
{passage_a}

PASSAGE B:
{passage_b}"""


def _normalize_q(q: str) -> str:
    return " ".join(q.lower().split())


def _parse_llm_json(text: str) -> dict | None:
    try:
        cleaned = re.sub(r"^```(?:json)?|```$", "", text.strip(), flags=re.MULTILINE).strip()
        return json.loads(cleaned)
    except Exception:
        return None


_DEGENERATE_MARKERS = (
    "this document", "this passage", "passage a", "passage b",
    "this section", "above", "the text", "the table", "both passages",
)


def _build_example(
    prompt: str,
    chunk_metas: list[tuple[str, str]],
    difficulty: str,
    seen: set[str],
) -> EvalExample | None:
    resp = llm.invoke([HumanMessage(content=prompt)])
    data = _parse_llm_json(resp.content)
    if not data:
        return None

    q = (data.get("question") or "").strip()
    snippet = (data.get("answer_snippet") or "").strip()
    ref_ans = (data.get("reference_answer") or "").strip() or None

    if len(q) < 12 or any(m in q.lower() for m in _DEGENERATE_MARKERS):
        return None

    nq = _normalize_q(q)
    if nq in seen:
        return None
    seen.add(nq)

    return EvalExample(
        question=q,
        gold_source_paths=[sp for sp, _ in chunk_metas],
        gold_snippets=[snippet] if snippet else [],
        reference_answer=ref_ans,
        difficulty=difficulty,
        notes=f"auto-gen ({difficulty}) from chunks {','.join(c for _, c in chunk_metas)}",
    )


def generate_balanced_eval_set(
    chunks: list[RagChunk],
    n_easy: int = 40,
    n_medium: int = 60,
    n_hard: int = 100,
    seed: int = 42,
    min_chunk_chars: int = 200,
    max_attempts_multiplier: float = 1.6,
    save_every: int = 20,
    save_path: Path | None = None,
) -> list[EvalExample]:
    rng = random.Random(seed)
    pool = [c for c in chunks if len(c.text) >= min_chunk_chars] or list(chunks)
    if not pool:
        raise ValueError("No suitable chunks in pool")

    examples: list[EvalExample] = []
    seen: set[str] = set()

    plan = [
        ("easy",   n_easy,   EASY_PROMPT,   1),
        ("medium", n_medium, MEDIUM_PROMPT, 1),
        ("hard",   n_hard,   HARD_PROMPT,   2),
    ]

    for diff, target, template, n_chunks in plan:
        log.info(f"--- {target} {diff.upper()} questions ---")
        produced = 0
        attempts = 0
        max_attempts = int(target * max_attempts_multiplier) + target
        pbar = tqdm(total=target, desc=f"{diff:<6}", unit="q")

        while produced < target and attempts < max_attempts:
            attempts += 1
            if n_chunks == 1:
                ch = rng.choice(pool)
                prompt = template.format(passage=ch.text[:2000])
                metas = [(ch.source_path, ch.chunk_id[:8])]
            else:
                ch_a = rng.choice(pool)
                others = [c for c in pool if c.source_path != ch_a.source_path]
                ch_b = rng.choice(others) if others else rng.choice(pool)
                prompt = template.format(passage_a=ch_a.text[:1500], passage_b=ch_b.text[:1500])
                metas = [(ch_a.source_path, ch_a.chunk_id[:8]),
                         (ch_b.source_path, ch_b.chunk_id[:8])]

            try:
                ex = _build_example(prompt, metas, diff, seen)
            except Exception as e:
                log.debug(f"{diff}: error {e}")
                continue
            if ex is None:
                continue

            examples.append(ex)
            produced += 1
            pbar.update(1)

            if save_path and produced % save_every == 0:
                save_eval_set(examples, save_path)

        pbar.close()
        if produced < target:
            log.warning(f"{diff}: only {produced}/{target} after {attempts} attempts")

    if save_path:
        save_eval_set(examples, save_path)

    counts = {d: sum(1 for e in examples if e.difficulty == d) for d in ("easy", "medium", "hard")}
    log.info(f"Done. Distribution: {counts}")
    return examples

In [65]:
# Delete the stale eval first
EVAL_DIR = PROJECT_ROOT / "eval"
EVAL_SET_PATH = EVAL_DIR / "eval_set.json"
EVAL_SET_PATH.unlink(missing_ok=True)

start = time.time()
eval_set = generate_balanced_eval_set(
    chunks=all_chunks,
    n_easy=20,
    n_medium=30,
    n_hard=50,
    seed=42,
    save_every=5,
    save_path=EVAL_SET_PATH,
)
elapsed = time.time() - start
log.info(f"Total time: {elapsed/60:.1f} min")

# Eyeball samples per difficulty
from collections import Counter
print("\nDistribution:", Counter(e.difficulty for e in eval_set))
print(f"With reference answers: {sum(1 for e in eval_set if e.reference_answer)}/{len(eval_set)}")

print("\nSample per difficulty:")
for diff in ("easy", "medium", "hard"):
    sample = next((e for e in eval_set if e.difficulty == diff), None)
    if sample:
        print(f"\n[{diff.upper()}] {sample.question}")
        print(f"  ref: {(sample.reference_answer or '—')[:160]}")
        print(f"  gold sources: {sample.gold_source_paths}")

2026-05-27 12:50:29,934 - INFO    | rag | --- 20 EASY questions ---


easy  :   0%|          | 0/20 [00:00<?, ?q/s]

2026-05-27 12:50:30,521 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:50:57,735 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:51:24,827 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:51:54,362 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:52:15,895 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:52:43,704 - INFO    | rag | Saved 5 examples → eval/eval_set.json
2026-05-27 12:52:44,259 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:53:11,684 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:53:39,313 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:53

medium:   0%|          | 0/30 [00:00<?, ?q/s]

2026-05-27 12:59:00,172 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:59:28,246 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 12:59:45,314 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:00:14,575 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:00:41,019 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:01:07,402 - INFO    | rag | Saved 25 examples → eval/eval_set.json
2026-05-27 13:01:07,947 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:01:37,064 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:02:10,410 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:0

hard  :   0%|          | 0/50 [00:00<?, ?q/s]

2026-05-27 13:13:56,892 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:14:43,127 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:15:21,990 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:16:09,184 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:16:57,053 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:17:43,943 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:18:26,555 - INFO    | rag | Saved 55 examples → eval/eval_set.json
2026-05-27 13:18:27,103 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:19:07,824 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:1

KeyboardInterrupt: 

In [ ]:
# EVAL_DIR = PROJECT_ROOT / "eval"
# EVAL_SET_PATH = EVAL_DIR / "eval_set.json"

# REGENERATE_EVAL = True  # flip to False AFTER you've curated the set

# if REGENERATE_EVAL or not EVAL_SET_PATH.exists():
#     eval_set = generate_eval_set(all_chunks, n=20, seed=42)
#     save_eval_set(eval_set, EVAL_SET_PATH)
# else:
#     eval_set = load_eval_set(EVAL_SET_PATH)
#     log.info(f"Loaded {len(eval_set)} curated examples")

# print(f"\n{len(eval_set)} eval examples — REVIEW THESE:\n")
# for i, ex in enumerate(eval_set, 1):
#     print(f"[{i}] ({ex.difficulty}) {ex.question}")
#     print(f"     gold : {ex.gold_source_paths}")
#     print(f"     snip : {ex.gold_snippets}\n")

In [66]:
import unicodedata


def _normalize(text: str) -> str:
    text = unicodedata.normalize("NFKC", text).lower()
    return re.sub(r"\s+", " ", text).strip()


def snippet_in_docs(snippet: str, docs: list[Document], min_overlap: float = 0.6) -> bool:
    """Gold snippet present in any retrieved doc: exact (normalized) OR token-overlap."""
    if not snippet:
        return False
    nsnip = _normalize(snippet)
    for d in docs:
        if nsnip in _normalize(d.page_content):
            return True
    snip_tokens = set(nsnip.split())
    if not snip_tokens:
        return False
    for d in docs:
        doc_tokens = set(_normalize(d.page_content).split())
        if len(snip_tokens & doc_tokens) / len(snip_tokens) >= min_overlap:
            return True
    return False


def evaluate_retriever(retrieve_fn, eval_set: list[EvalExample], k: int = 5) -> dict:
    """Hit@k, Recall@k, MRR, Snippet-Hit@k for a single retriever."""
    n = hits = snippet_hits = 0
    recall_sum = rr_sum = 0.0
    per_diff = defaultdict(lambda: {"n": 0, "hit": 0})

    for ex in eval_set:
        gold = set(ex.gold_source_paths)
        if not gold:
            continue  # negatives belong to the generation/threshold eval
        n += 1

        docs = retrieve_fn(ex.question, k)
        sources = [d.metadata.get("source_path") for d in docs]

        hit = any(s in gold for s in sources)
        hits += int(hit)

        recall_sum += len({s for s in sources if s in gold}) / len(gold)

        rr = 0.0
        for rank, s in enumerate(sources, 1):
            if s in gold:
                rr = 1.0 / rank
                break
        rr_sum += rr

        if any(snippet_in_docs(sn, docs) for sn in ex.gold_snippets):
            snippet_hits += 1

        per_diff[ex.difficulty]["n"] += 1
        per_diff[ex.difficulty]["hit"] += int(hit)

    if n == 0:
        return {"n": 0}

    return {
        "n": n,
        "hit@k": hits / n,
        "recall@k": recall_sum / n,
        "mrr": rr_sum / n,
        "snippet_hit@k": snippet_hits / n,
        "by_difficulty": {
            d: round(v["hit"] / v["n"], 3)
            for d, v in sorted(per_diff.items()) if v["n"]
        },
    }

In [68]:
# Uniform adapters: (query, k) -> list[Document]
RETRIEVERS = {
    "dense":           lambda q, k: [d for d, _ in vectorstore.similarity_search_with_score(q, k=k)],
    "bm25":            lambda q, k: [d for d, _ in bm25_retriever.retrieve(q, k=k)],
    "hybrid":          lambda q, k: [d for d, _ in ensemble.retrieve(q, top_k=k)],
    "hybrid_reranked": lambda q, k: [d for d, _ in hybrid_reranked.retrieve(q, top_k=k)],
}

EVAL_K = cfg.TOP_K
eval_set = load_eval_set(EVAL_SET_PATH)
n_pos = len([e for e in eval_set if e.gold_source_paths])
print(f"Evaluating {n_pos} positive examples @ k={EVAL_K}\n")

results = {}
for name, fn in RETRIEVERS.items():
    log.info(f"Evaluating: {name}")
    results[name] = evaluate_retriever(fn, eval_set, k=EVAL_K)

print(f"\n{'Retriever':<18}{'Hit@k':>8}{'Recall@k':>10}{'MRR':>8}{'Snippet@k':>11}")
print("-" * 55)
for name, r in results.items():
    if r.get("n", 0) == 0:
        continue
    print(f"{name:<18}{r['hit@k']:>8.3f}{r['recall@k']:>10.3f}"
          f"{r['mrr']:>8.3f}{r['snippet_hit@k']:>11.3f}")
print("-" * 55)

print("\nHit@k by difficulty:")
for name, r in results.items():
    if r.get("n", 0):
        print(f"  {name:<18} {r['by_difficulty']}")

# Persist for regression history
RESULTS_DIR = EVAL_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
run_path = RESULTS_DIR / f"retrieval_{datetime.now(timezone.utc):%Y%m%d_%H%M%S}.json"
run_path.write_text(json.dumps(results, indent=2), encoding="utf-8")
log.info(f"Saved run → {run_path.relative_to(PROJECT_ROOT)}")

2026-05-27 13:47:02,991 - INFO    | rag | Evaluating: dense


Evaluating 90 positive examples @ k=5



2026-05-27 13:47:04,050 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,122 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,188 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,263 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,328 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,394 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,467 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,533 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:47:04,601 - INFO    | httpx | HTTP Request: POST http://localhost:11434/ap


Retriever            Hit@k  Recall@k     MRR  Snippet@k
-------------------------------------------------------
dense                0.856     0.700   0.564      0.422
bm25                 0.744     0.594   0.519      0.389
hybrid               0.856     0.700   0.646      0.433
hybrid_reranked      0.956     0.778   0.724      0.433
-------------------------------------------------------

Hit@k by difficulty:
  dense              {'easy': 0.9, 'hard': 0.75, 'medium': 0.967}
  bm25               {'easy': 0.95, 'hard': 0.775, 'medium': 0.567}
  hybrid             {'easy': 1.0, 'hard': 0.825, 'medium': 0.8}
  hybrid_reranked    {'easy': 1.0, 'hard': 0.95, 'medium': 0.933}


# RAG Retriever Evaluation Report

## Overall Retrieval Performance

| Retriever | Hit@k ↑ | Recall@k ↑ | MRR ↑ | Snippet@k ↑ |
|------------|----------|-------------|--------|--------------|
| Dense Retrieval | 0.856 | 0.700 | 0.564 | 0.422 |
| BM25 Retrieval | 0.744 | 0.594 | 0.519 | 0.389 |
| Hybrid Retrieval | 0.856 | 0.700 | 0.646 | 0.433 |
| Hybrid + Reranker | **0.956** | **0.778** | **0.724** | **0.433** |

---

# What Do These Metrics Mean?

## 1. Hit@k

### Definition
Measures whether the correct/relevant document appears within the top-k retrieved results.

### Formula

$\text{Hit@k} = \frac{ \text{Number of queries with at least one relevant result in top-k}}{\text{Total number of queries}}$

### Example

If:
- 100 queries are evaluated
- 95 queries successfully retrieve at least one correct document

Then:

$
\text{Hit@k} = \frac{95}{100} = 0.95
$

### Interpretation

| Score Range | Meaning |
|--------------|----------|
| 1.0 | Perfect retrieval |
| > 0.90 | Excellent |
| 0.70 – 0.90 | Good |
| < 0.60 | Weak retrieval |

### In Our Results

| Retriever | Hit@k |
|------------|--------|
| Hybrid + Reranker | **0.956** |

This means:
- the system successfully retrieves the correct document for ~96% of queries.

---

# 2. Recall@k

## Definition
Measures how many of all relevant documents are retrieved within the top-k results.

## Formula


$\text{Recall@k} =
\frac{
\text{Relevant documents retrieved in top-k}
}{
\text{Total relevant documents}
}$


## Example

Suppose:
- Total relevant documents = 5
- Retrieved relevant documents = 4

Then:


$\text{Recall@k} = \frac{4}{5} = 0.8$


## Interpretation

| Score | Meaning |
|--------|----------|
| High Recall | Retriever captures most useful context |
| Low Recall | Important information is missing |

## In Our Results

| Retriever | Recall@k |
|------------|------------|
| Hybrid + Reranker | **0.778** |

Meaning:
- ~78% of all relevant information is successfully retrieved.

---

# 3. MRR (Mean Reciprocal Rank)

## Definition
Measures how early the first relevant result appears in ranking.

Higher MRR means:
- relevant documents appear closer to the top,
- improving downstream LLM response quality.

## Formula


$\text{MRR} =
\frac{1}{|Q|}
\sum_{i=1}^{|Q|}
\frac{1}{\text{Rank}_i}$


Where:
- \(Q\) = number of queries
- Rank = position of first relevant document

---

## Example

| First Relevant Rank | Reciprocal Rank |
|----------------------|----------------|
| Rank 1 | 1.0 |
| Rank 2 | 0.5 |
| Rank 5 | 0.2 |

Average of all reciprocal ranks gives MRR.

---

## Interpretation

| MRR Score | Meaning |
|------------|----------|
| 1.0 | Relevant document always ranked first |
| > 0.70 | Excellent ranking quality |
| 0.50 – 0.70 | Good |
| < 0.40 | Weak ranking |

---

## In Our Results

| Retriever | MRR |
|------------|------|
| Hybrid + Reranker | **0.724** |

This indicates:
- relevant documents are usually ranked near the top.

---

# 4. Snippet@k

## Definition
Measures whether the retrieved chunk/snippet directly contains the answer.

This metric evaluates:
- answer localization quality,
- chunking effectiveness.

## Formula

$
\text{Snippet@k} =
\frac{
\text{Queries where retrieved snippet contains answer}
}{
\text{Total queries}
}$


---

## Interpretation

| High Snippet@k | Low Snippet@k |
|----------------|----------------|
| Retrieved chunks contain direct evidence | Retrieved docs are related but may not contain exact answers |

---

## In Our Results

| Retriever | Snippet@k |
|------------|------------|
| Hybrid + Reranker | **0.433** |

This suggests:
- retrieval is strong,
- but chunk-level answer localization can still improve.

Possible improvements:
- better chunking,
- semantic chunk boundaries,
- overlap tuning.

---

# Retrieval Performance by Query Difficulty

| Retriever | Easy | Medium | Hard |
|------------|------|---------|------|
| Dense Retrieval | 0.900 | 0.967 | 0.750 |
| BM25 Retrieval | 0.950 | 0.567 | 0.775 |
| Hybrid Retrieval | 1.000 | 0.800 | 0.825 |
| Hybrid + Reranker | **1.000** | **0.933** | **0.950** |

---

# Understanding Query Difficulty

| Difficulty | Description |
|-------------|-------------|
| Easy | Strong keyword overlap |
| Medium | Partial paraphrasing / semantic variation |
| Hard | Complex semantic reasoning / low lexical overlap |

---

# Key Observations

## Dense Retrieval
- Strong semantic understanding
- Performs well on medium queries
- Slightly weaker on hard queries

---

## BM25 Retrieval
- Excellent lexical matching
- Weak semantic generalization
- Poor medium-query performance

---

## Hybrid Retrieval
- Combines lexical + semantic retrieval
- More balanced performance
- Stronger robustness

---

## Hybrid + Reranker
- Best overall system
- Highest retrieval accuracy
- Best ranking quality
- Strongest hard-query performance

---

# Final Conclusion

## Best Performing Pipeline


$\boxed{
\text{Hybrid Retrieval + Reranking}
}$


because it provides:
- highest Hit@k,
- highest Recall@k,
- highest MRR,
- strongest hard-query retrieval.

---

# Practical Impact on RAG Quality

| Metric Improvement | Real-World Effect |
|--------------------|-------------------|
| Higher Hit@k | Fewer missed answers |
| Higher Recall@k | Better context coverage |
| Higher MRR | Better top-ranked evidence |
| Higher Snippet@k | More grounded responses |

---

# Overall Recommendation

The evaluation strongly suggests that:

```text
Hybrid Retrieval + Reranking
=
Most reliable production-grade RAG architecture
```

because it effectively combines:
- semantic understanding,
- keyword matching,
- intelligent ranking refinement.

In [ ]:
# token evaluation
# ipc
# iso COMPLIANCE 
# US legal database

: 

In [ ]:
#pip install "ragas>=0.2" datasets nest_asyncio --break-system-packages

: 

In [69]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning, module="ragas")

import nest_asyncio
nest_asyncio.apply()

from ragas import EvaluationDataset, evaluate
from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
)
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
]

run_config = RunConfig(timeout=180, max_retries=3, max_wait=60, max_workers=1)

log.info("RAGAS ready | judge: local Gemma | metrics: Faithfulness, ResponseRelevancy, ContextPrecision")



/tmp/ipykernel_1979552/500254689.py:8: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_1979552/500254689.py:8: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
/tmp/ipykernel_1979552/500254689.py:8: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import (
/tmp/ipykernel_1979552/500254689.py:17: DeprecationWarning: LangchainLLMWrapper is deprecate

In [70]:
positives = [e for e in eval_set if e.gold_source_paths]

# Smoke-start: 5 rows first. Bump to len(positives) once you've verified it runs.
N_EVAL = 5

ragas_rows = []
for ex in tqdm(positives[:N_EVAL], desc="Generating answers"):
    # Single retrieval pass → reuse for both context list and answer generation
    results = hybrid_reranked.retrieve(ex.question, top_k=cfg.TOP_K)
    contexts = [d.page_content for d, _ in results]

    context_block, _ = build_context(results)
    user_msg = f"CONTEXT:\n{context_block}\n\nQUESTION: {ex.question}\n\nANSWER:"
    resp = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=user_msg),
    ])

    ragas_rows.append({
        "user_input": ex.question,
        "retrieved_contexts": contexts,
        "response": resp.content,
        "reference": ex.gold_snippets[0] if ex.gold_snippets else "",
    })

ragas_dataset = EvaluationDataset.from_list(ragas_rows)
log.info(f"RAGAS dataset built: {len(ragas_dataset)} rows")

Generating answers:   0%|          | 0/5 [00:00<?, ?it/s]

2026-05-27 13:49:12,938 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:49:19,308 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:49:44,058 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:49:45,963 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:50:04,817 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:50:06,638 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:50:33,205 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 13:50:35,241 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 13:50:55,881 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/em

In [76]:
import json

def load_eval_set(path):
    with open(path, "r") as f:
        return json.load(f)

eval_set = load_eval_set(EVAL_SET_PATH)

print("Total samples:", len(eval_set))
print(eval_set[0].keys())

Total samples: 90
dict_keys(['question', 'gold_source_paths', 'gold_snippets', 'reference_answer', 'difficulty', 'notes'])


In [80]:
from tqdm import tqdm

from ragas import EvaluationDataset, evaluate

from ragas.metrics import (
    Faithfulness,
    ResponseRelevancy,
    LLMContextPrecisionWithoutReference,
    SemanticSimilarity,
    AnswerCorrectness,
)

from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper


/tmp/ipykernel_1979552/2647465705.py:5: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import (
/tmp/ipykernel_1979552/2647465705.py:5: DeprecationWarning: Importing ResponseRelevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import ResponseRelevancy
  from ragas.metrics import (
/tmp/ipykernel_1979552/2647465705.py:5: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import (
/tmp/ipykernel_1979552/2647465705.py:5: DeprecationWarning: Importing SemanticSimilarity 

In [81]:
ragas_llm = LangchainLLMWrapper(llm)
ragas_emb = LangchainEmbeddingsWrapper(embeddings)

/tmp/ipykernel_1979552/2167887831.py:1: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(llm)
/tmp/ipykernel_1979552/2167887831.py:2: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_emb = LangchainEmbeddingsWrapper(embeddings)


In [82]:
metrics = [
    Faithfulness(llm=ragas_llm),
    ResponseRelevancy(llm=ragas_llm, embeddings=ragas_emb),
    LLMContextPrecisionWithoutReference(llm=ragas_llm),
    SemanticSimilarity(embeddings=ragas_emb),
    AnswerCorrectness(llm=ragas_llm, embeddings=ragas_emb),
]

In [83]:
ragas_rows = []

for ex in tqdm(eval_set[:90]):   # start with full set later

    # --------------------------
    # Retrieval
    # --------------------------
    results = hybrid_reranked.retrieve(
        ex["question"],
        top_k=5
    )

    contexts = [d.page_content for d, _ in results]

    context_block = "\n\n".join(contexts)

    # --------------------------
    # Generation
    # --------------------------
    prompt = f"""
CONTEXT:
{context_block}

QUESTION:
{ex['question']}

ANSWER:
"""

    response = llm.invoke(prompt)

    response_text = (
        response.content
        if hasattr(response, "content")
        else str(response)
    )

    # --------------------------
    # Build RAGAS row
    # --------------------------
    ragas_rows.append({
        "user_input": ex["question"],
        "retrieved_contexts": contexts,
        "response": response_text,
        "reference": ex["reference_answer"],
    })


2026-05-27 14:30:53,399 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 14:30:59,791 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

2026-05-27 14:31:28,370 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 14:31:30,159 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

2026-05-27 14:31:41,365 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 14:31:43,122 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

2026-05-27 14:31:56,082 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 14:31:57,958 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"

2026-05-27 14:32:07,110 - INFO    | httpx | HTTP Request: POST http://localhost:11434/a

In [84]:
dataset = EvaluationDataset.from_list(ragas_rows)

print("Dataset size:", len(dataset))

Dataset size: 90


In [88]:
results = evaluate(
    dataset=dataset,
    metrics=metrics,
)

df = results.to_pandas()
df.head()

Evaluating:   0%|          | 0/450 [00:00<?, ?it/s]

2026-05-27 18:42:06,490 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 18:42:16,517 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 18:42:16,519 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 18:42:16,521 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 18:42:36,571 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
2026-05-27 18:42:37,722 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 18:42:47,751 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 18:42:47,752 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/embed "HTTP/1.1 200 OK"
2026-05-27 18:42:47,753 - INFO    | httpx | HTTP Request: POST http://localhost:11434/api/

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,llm_context_precision_without_reference,semantic_similarity,answer_correctness
0,"According to Section 283, what is the fine for...",[CHAPTER XV\n- 282. Rash navigation of vessel....,The fine for danger or obstruction in public w...,Section 283,NaN,0.764585,NaN,0.160970,NaN
1,"According to Section 135, what is the maximum ...","[of such assault, if the assault committed. Ab...",The maximum imprisonment term for abetment of ...,imprisonment of either description a term whic...,NaN,NaN,NaN,0.638935,NaN
2,"According to Section 109, what is the rule reg...",[Illustration.\nA concerts with B a plan f...,"According to Section 109, if the act abetted i...",Section 109,NaN,NaN,NaN,0.596578,NaN
3,What is the maximum imprisonment period for cr...,[1*[CHAPTER XXA\nOF CRUELTY BY HUSBAND OR RELA...,The maximum imprisonment period for cruelty un...,for a term which may extend to three years and...,NaN,NaN,NaN,0.619763,NaN
4,"According to Section 457, what is the maximum ...",[or wrongful restraint. assault\n455. Lurking ...,"Based on Section 457, if the intended offence ...",The term of imprisonment may be extended to fo...,NaN,NaN,NaN,0.963554,NaN


In [89]:
print("\n===== AVERAGE METRICS =====\n")

exclude_cols = {
    "user_input",
    "response",
    "retrieved_contexts",
    "reference",
}

for col in df.columns:
    if col not in exclude_cols:
        print(f"{col}: {df[col].mean():.4f}")


===== AVERAGE METRICS =====

faithfulness: nan
answer_relevancy: 0.6837
llm_context_precision_without_reference: nan
semantic_similarity: 0.6108
answer_correctness: nan


In [90]:
df["difficulty"] = [ex["difficulty"] for ex in eval_set[:len(df)]]

print(df.groupby("difficulty").mean(numeric_only=True))

            faithfulness  answer_relevancy  \
difficulty                                   
easy                 NaN          0.651383   
hard                 NaN               NaN   
medium               NaN          0.780490   

            llm_context_precision_without_reference  semantic_similarity  \
difficulty                                                                 
easy                                            NaN             0.540509   
hard                                            NaN             0.579115   
medium                                          NaN             0.700007   

            answer_correctness  
difficulty                      
easy                       NaN  
hard                       NaN  
medium                     NaN  


In [ ]:
#!pip install "datasets<4.0" --break-system-packages  # RAGAS currently incompatible with datasets v4

In [ ]:
# # Example: 50 EU policy summaries via HuggingFace
# from datasets import load_dataset
# ds = load_dataset("FiscalNote/billsum", split="train[:50]")
# # then write each .text to data/raw/text/eurlex_001.txt etc.
# from pathlib import Path
# out = Path("../data/raw/pdfs/")
# out.mkdir(parents=True, exist_ok=True)
# for i, row in enumerate(ds):
#     (out / f"bill_{i:03d}.txt").write_text(row["text"], encoding="utf-8")
# print(f"Wrote {len(ds)} files to {out}/")

In [ ]:
# rm -rf chroma_db/
# rm -f data/processed/*.json
# rm -f eval/eval_set.json
# rm -rf eval/results/